# Acoustic Machine-Fault Detection — Local Runtime Edition

This notebook is adapted from your uploaded capstone notebook PDF.

## What changed
- Removed Google Drive mounting and `/content/...` paths.
- Reads MIMII WAV files directly from your Windows PC through a **Colab local runtime**.
- Processes all 12 combinations: fan, valve, slider, pump × −6 dB, 0 dB, +6 dB.
- Saves extracted features, metadata, model checkpoints, and evaluation outputs locally under `PROJECT`.
- Uses CPU automatically when CUDA/NVIDIA GPU is unavailable.

**Important:** This notebook runs on your PC, not Google's hosted runtime. Connect Colab to your local Jupyter runtime before running the cells. Keep your PC awake during extraction/training.

## 1. Set your local dataset path

First extract your ZIP files on Windows. The directory you enter below should contain folders such as `-6_dB_fan`, `0_dB_fan`, `6_dB_fan`, `-6_dB_valve`, etc.

Change `PROJECT` to the real folder on your PC. Use a raw string (`r"..."`) for Windows paths.

In [ ]:
# EDIT THIS PATH to the folder containing the 12 extracted dataset folders.
# Example: r"D:\MIMII_Dataset"
PROJECT = r"D:\MIMII_Dataset"

import os
from pathlib import Path

PROJECT = str(Path(PROJECT).expanduser())
os.makedirs(PROJECT, exist_ok=True)

print("Project folder:", PROJECT)
print("Folder exists:", os.path.isdir(PROJECT))
if os.path.isdir(PROJECT):
    print("Items:", os.listdir(PROJECT)[:40])

## 2. Install/import dependencies

Run this cell once after connecting to your local runtime. If packages install successfully but imports fail, restart the local Jupyter kernel and rerun from the top.

In [ ]:
%pip -q install librosa soundfile scikit-learn matplotlib pandas tqdm torch

In [ ]:
import os
import random
import time
import json
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch device:", DEVICE)
if DEVICE.type == "cpu":
    print("CPU mode: start with fewer files per class to check runtime.")

## 3. Dataset and feature settings

This notebook extracts features for all 12 dataset folders. To keep the first run manageable on a CPU-only PC, the default is 100 files per class per condition (up to 200 files per dataset folder). Increase this later if your PC has enough RAM and time.

In [ ]:
MACHINES = ["fan", "valve", "slider", "pump"]
CONDITIONS = ["-6_dB", "0_dB", "6_dB"]
DATASETS = [f"{condition}_{machine}" for machine in MACHINES for condition in CONDITIONS]

SR = 16000
DURATION = 5
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 512
MAX_PER_CLASS_PER_CONDITION = 100

print("Datasets:", len(DATASETS))
print("\n".join(DATASETS))
print("Maximum selected files per dataset:", MAX_PER_CLASS_PER_CONDITION * 2)

In [ ]:
def waveform_to_logmel(audio):
    target_length = int(SR * DURATION)
    audio = np.asarray(audio, dtype=np.float32)

    if len(audio) < target_length:
        audio = np.pad(audio, (0, target_length - len(audio)))
    else:
        audio = audio[:target_length]

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = np.clip(log_mel, -80, 0)
    log_mel = (log_mel + 80) / 80
    return log_mel.astype(np.float32)


def audio_to_logmel(file_path):
    audio, _ = librosa.load(
        file_path, sr=SR, mono=True, duration=DURATION
    )
    return waveform_to_logmel(audio)


def label_from_member(member):
    parts = member.lower().replace("\\", "/").split("/")
    if "normal" in parts:
        return 0
    if "abnormal" in parts:
        return 1
    return None

print("Audio preprocessing functions ready.")

## 4. Check local WAV files before feature extraction

Run this first. If a dataset shows zero WAV files, check that its ZIP was fully extracted and that the path points to the parent folder containing the dataset folders.

In [ ]:
def count_wavs(folder):
    count = 0
    if not os.path.isdir(folder):
        return 0
    for root, dirs, files in os.walk(folder):
        count += sum(f.lower().endswith(".wav") for f in files)
    return count

dataset_inventory = []
for dataset_name in DATASETS:
    folder = os.path.join(PROJECT, dataset_name)
    n_wavs = count_wavs(folder)
    dataset_inventory.append({
        "dataset": dataset_name,
        "folder_exists": os.path.isdir(folder),
        "wav_files": n_wavs
    })

inventory_df = pd.DataFrame(dataset_inventory)
display(inventory_df)
print("Total WAV files found:", inventory_df["wav_files"].sum())

## 5. Extract Log-Mel features for all 12 datasets

Features and metadata are saved separately per dataset. Missing or empty folders are skipped with a clear message. The cell also avoids silently saving an empty feature array.

In [ ]:
for dataset_name in DATASETS:
    print("\n" + "=" * 60)
    print("Processing:", dataset_name)

    dataset_dir = os.path.join(PROJECT, dataset_name)
    output_path = os.path.join(PROJECT, dataset_name + "_features.npz")
    metadata_path = os.path.join(PROJECT, dataset_name + "_metadata.csv")

    if os.path.exists(output_path) and os.path.exists(metadata_path):
        print("Feature files already exist; skipping:", dataset_name)
        continue

    if not os.path.isdir(dataset_dir):
        print("Dataset folder missing; skipping:", dataset_dir)
        continue

    wav_files = []
    for root, dirs, files in os.walk(dataset_dir):
        for file in files:
            if file.lower().endswith(".wav"):
                wav_files.append(os.path.join(root, file))

    if not wav_files:
        print("No WAV files found; skipping:", dataset_name)
        continue

    class_members = {0: [], 1: []}
    for filepath in wav_files:
        rel_path = os.path.relpath(filepath, dataset_dir)
        label = label_from_member(rel_path)
        if label is not None:
            class_members[label].append((filepath, rel_path))

    print("Available normal:", len(class_members[0]))
    print("Available abnormal:", len(class_members[1]))

    selected = []
    for label in (0, 1):
        candidates = sorted(class_members[label])
        random.Random(SEED + label).shuffle(candidates)
        selected.extend([
            (filepath, rel_path, label)
            for filepath, rel_path in candidates[:MAX_PER_CLASS_PER_CONDITION]
        ])

    if not selected:
        print("No files inside normal/abnormal class folders; skipping.")
        continue

    features, labels, records = [], [], []
    for i, (filepath, rel_path, label) in enumerate(
        tqdm(selected, desc=dataset_name), start=1
    ):
        try:
            audio, _ = librosa.load(
                filepath, sr=SR, mono=True, duration=DURATION
            )
            feature = waveform_to_logmel(audio)
            features.append(feature)
            labels.append(label)

            parts = rel_path.replace("\\", "/").split("/")
            machine_id = next(
                (p for p in parts if p.startswith("id_")), "unknown"
            )
            machine = next(
                (m for m in MACHINES if dataset_name.endswith("_" + m)),
                "unknown"
            )
            condition = dataset_name[:-(len(machine) + 1)]

            records.append({
                "dataset": dataset_name,
                "machine": machine,
                "condition": condition,
                "machine_id": machine_id,
                "file": rel_path,
                "label": label
            })
        except Exception as exc:
            print("Skipped:", rel_path, "|", str(exc)[:150])

    if not features:
        print("No usable audio features; skipping:", dataset_name)
        continue

    X_condition = np.stack(features).astype(np.float32)
    y_condition = np.asarray(labels, dtype=np.int64)
    metadata_condition = pd.DataFrame(records)

    np.savez_compressed(output_path, X=X_condition, y=y_condition)
    metadata_condition.to_csv(metadata_path, index=False)

    print("Saved feature shape:", X_condition.shape)
    print("Saved labels:", y_condition.shape)
    print("Saved metadata:", metadata_path)

print("\nFeature extraction pass complete.")

## 6. Combine saved features and create train/validation/test splits

Only successfully generated feature files are included. Splitting is stratified by normal/abnormal label. For rigorous machine-generalization evaluation, a later improvement is to split by machine ID or source recording rather than random clips.

In [ ]:
X_parts, y_parts, metadata_parts = [], [], []

for dataset_name in DATASETS:
    feature_path = os.path.join(PROJECT, dataset_name + "_features.npz")
    metadata_path = os.path.join(PROJECT, dataset_name + "_metadata.csv")

    if not (os.path.exists(feature_path) and os.path.exists(metadata_path)):
        print("Skipping missing feature output:", dataset_name)
        continue

    with np.load(feature_path) as saved:
        X_parts.append(saved["X"])
        y_parts.append(saved["y"])
    metadata_parts.append(pd.read_csv(metadata_path))

if not X_parts:
    raise FileNotFoundError(
        "No feature files were found. Check PROJECT, extracted WAV folders, and run Step 5."
    )

X = np.concatenate(X_parts).astype(np.float32)
y = np.concatenate(y_parts).astype(np.int64)
metadata = pd.concat(metadata_parts, ignore_index=True)

assert len(X) == len(y) == len(metadata)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Class counts:")
print(pd.Series(y).map({0: "normal", 1: "abnormal"}).value_counts())
print("\nSamples by machine:")
display(metadata["machine"].value_counts().rename_axis("machine").to_frame("samples"))
print("\nSamples by condition:")
display(metadata["condition"].value_counts().rename_axis("condition").to_frame("samples"))

In [ ]:
all_indices = np.arange(len(y))
train_idx, remaining_idx = train_test_split(
    all_indices, test_size=0.30, random_state=SEED, stratify=y
)
val_idx, test_idx = train_test_split(
    remaining_idx, test_size=0.50, random_state=SEED,
    stratify=y[remaining_idx]
)

np.save(os.path.join(PROJECT, "train_idx.npy"), train_idx)
np.save(os.path.join(PROJECT, "val_idx.npy"), val_idx)
np.save(os.path.join(PROJECT, "test_idx.npy"), test_idx)
metadata.to_csv(os.path.join(PROJECT, "all_metadata.csv"), index=False)

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

sample_idx = int(train_idx[0])
plt.figure(figsize=(10, 4))
plt.imshow(X[sample_idx], origin="lower", aspect="auto", cmap="magma")
plt.colorbar(label="Scaled log-Mel value")
plt.xlabel("Time frames")
plt.ylabel("Mel frequency bins")
plt.title(
    f"{metadata.iloc[sample_idx]['machine']} audio — "
    f"{'Abnormal' if y[sample_idx] == 1 else 'Normal'}"
)
plt.tight_layout()
plt.show()

## 7. Create PyTorch data loaders

In [ ]:
BATCH_SIZE = 16
MAX_EPOCHS = 30
PATIENCE = 5
LEARNING_RATE = 1e-3

def make_loader(indices, shuffle=False):
    features = torch.from_numpy(X[indices]).float().unsqueeze(1)
    labels = torch.from_numpy(y[indices]).float()
    dataset = TensorDataset(features, labels)
    return DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0
    )

train_loader = make_loader(train_idx, shuffle=True)
val_loader = make_loader(val_idx)
test_loader = make_loader(test_idx)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

## 8. Train the baseline CNN

In [ ]:
class FanCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.30)
        self.fc = nn.Linear(64, 1)

    def forward(self, x, return_features=False):
        x = self.conv1(x)
        x = self.conv2(x)
        features = self.conv3(x)
        pooled = self.pool(features).flatten(1)
        logits = self.fc(self.dropout(pooled)).squeeze(1)
        if return_features:
            return logits, features
        return logits


def train_model(model, train_loader, val_loader, checkpoint_path):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.BCEWithLogitsLoss()

    best_auc = -1.0
    best_loss = float("inf")
    patience_count = 0
    history = []

    for epoch in range(MAX_EPOCHS):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses, val_labels, val_probs = [], [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_losses.append(loss.item())
                val_labels.extend(yb.cpu().numpy())
                val_probs.extend(torch.sigmoid(logits).cpu().numpy())

        val_auc = roc_auc_score(val_labels, val_probs) if len(np.unique(val_labels)) == 2 else float("nan")
        val_loss = float(np.mean(val_losses))
        train_loss = float(np.mean(train_losses))
        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_auc": val_auc
        })
        print(
            f"Epoch {epoch+1:02d}/{MAX_EPOCHS} | "
            f"train loss={train_loss:.4f} | val loss={val_loss:.4f} | "
            f"val AUC={val_auc:.4f}"
        )

        improved = (
            val_auc > best_auc + 1e-4
            or (abs(val_auc - best_auc) <= 1e-4 and val_loss < best_loss)
        )
        if improved:
            best_auc, best_loss = val_auc, val_loss
            patience_count = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            patience_count += 1

        if patience_count >= PATIENCE:
            print("Early stopping.")
            break

    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    return model, pd.DataFrame(history)


CNN_PATH = os.path.join(PROJECT, "cnn_best.pt")
cnn = FanCNN()
cnn, cnn_history = train_model(cnn, train_loader, val_loader, CNN_PATH)
cnn_history.to_csv(os.path.join(PROJECT, "cnn_training_history.csv"), index=False)
print("CNN checkpoint:", CNN_PATH)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(cnn_history["epoch"], cnn_history["train_loss"], label="Train loss")
axes[0].plot(cnn_history["epoch"], cnn_history["val_loss"], label="Validation loss")
axes[0].set_title("CNN loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(cnn_history["epoch"], cnn_history["val_auc"], label="Validation AUC")
axes[1].set_title("CNN validation ROC-AUC")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

## 9. Train the CNN-GRU model

In [ ]:
class FanCNNGRU(nn.Module):
    def __init__(self, hidden_size=64):
        super().__init__()
        self.conv1 = FanCNN().conv1
        self.conv2 = FanCNN().conv2
        self.conv3 = FanCNN().conv3
        self.gru = nn.GRU(
            input_size=64, hidden_size=hidden_size, batch_first=True
        )
        self.dropout = nn.Dropout(0.30)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = x.mean(dim=2)          # (batch, channels, time)
        x = x.transpose(1, 2)      # (batch, time, channels)
        sequence, _ = self.gru(x)
        final_state = sequence[:, -1, :]
        return self.fc(self.dropout(final_state)).squeeze(1)


GRU_PATH = os.path.join(PROJECT, "cnn_gru_best.pt")
cnn_gru = FanCNNGRU()
cnn_gru, gru_history = train_model(cnn_gru, train_loader, val_loader, GRU_PATH)
gru_history.to_csv(os.path.join(PROJECT, "cnn_gru_training_history.csv"), index=False)
print("CNN-GRU checkpoint:", GRU_PATH)

plt.figure(figsize=(8, 4))
plt.plot(cnn_history["epoch"], cnn_history["val_auc"], label="CNN validation AUC")
plt.plot(gru_history["epoch"], gru_history["val_auc"], label="CNN-GRU validation AUC")
plt.xlabel("Epoch")
plt.ylabel("ROC-AUC")
plt.title("Validation performance comparison")
plt.legend()
plt.grid(True)
plt.show()

## 10. Evaluation helpers and overall test results

In [ ]:
def predict_probabilities(model, features, batch_size=32):
    model.eval()
    results = []
    loader = DataLoader(
        TensorDataset(torch.from_numpy(np.asarray(features, dtype=np.float32)).unsqueeze(1)),
        batch_size=batch_size, shuffle=False, num_workers=0
    )
    with torch.no_grad():
        for (xb,) in loader:
            logits = model(xb.to(DEVICE))
            results.extend(torch.sigmoid(logits).cpu().numpy())
    return np.asarray(results).reshape(-1)


def evaluate_model(model, X_test, y_test, threshold=0.5):
    start_time = time.perf_counter()
    probabilities = predict_probabilities(model, X_test)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start_time

    y_test = np.asarray(y_test).reshape(-1).astype(int)
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, predictions, labels=[0, 1]).ravel()

    metrics = {
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities) if len(np.unique(y_test)) == 2 else np.nan,
        "false_alarm_rate": fp / max(tn + fp, 1),
        "mean_latency_ms_per_clip": elapsed * 1000 / max(len(y_test), 1)
    }
    return metrics, probabilities, predictions


X_test = X[test_idx]
y_test = y[test_idx]

cnn_metrics, cnn_probs, cnn_preds = evaluate_model(cnn, X_test, y_test)
gru_metrics, gru_probs, gru_preds = evaluate_model(cnn_gru, X_test, y_test)

comparison = pd.DataFrame([cnn_metrics, gru_metrics], index=["CNN", "CNN-GRU"])
display(comparison.round(4))
comparison.to_csv(os.path.join(PROJECT, "model_comparison.csv"))

np.save(os.path.join(PROJECT, "cnn_test_probabilities.npy"), cnn_probs)
np.save(os.path.join(PROJECT, "cnn_test_predictions.npy"), cnn_preds)
np.save(os.path.join(PROJECT, "gru_test_probabilities.npy"), gru_probs)
np.save(os.path.join(PROJECT, "gru_test_predictions.npy"), gru_preds)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, preds, name in [(axes[0], cnn_preds, "CNN"), (axes[1], gru_preds, "CNN-GRU")]:
    cm = confusion_matrix(y_test, preds, labels=[0, 1])
    ax.imshow(cm, cmap="Blues")
    ax.set_title(name)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks([0, 1], ["Normal", "Abnormal"])
    ax.set_yticks([0, 1], ["Normal", "Abnormal"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.tight_layout()
plt.show()

print("CNN classification report:")
print(classification_report(y_test, cnn_preds, target_names=["Normal", "Abnormal"], zero_division=0))
print("CNN-GRU classification report:")
print(classification_report(y_test, gru_preds, target_names=["Normal", "Abnormal"], zero_division=0))

## 11. ROC curve and results by machine/noise condition

In [ ]:
plt.figure(figsize=(7, 5))
for probs, name in [(cnn_probs, "CNN"), (gru_probs, "CNN-GRU")]:
    if len(np.unique(y_test)) == 2:
        fpr, tpr, _ = roc_curve(y_test, probs)
        auc_value = roc_auc_score(y_test, probs)
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc_value:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random baseline")
plt.xlabel("False-positive rate")
plt.ylabel("True-positive rate")
plt.title("ROC curve — machine-fault detection")
plt.legend()
plt.grid(True)
plt.show()

test_metadata = metadata.iloc[test_idx].reset_index(drop=True)
condition_results = []

for machine in MACHINES:
    for condition in CONDITIONS:
        mask = (
            (test_metadata["machine"].to_numpy() == machine)
            & (test_metadata["condition"].to_numpy() == condition)
        )
        if mask.sum() == 0:
            continue

        y_cond = y_test[mask]
        for model_name, probabilities in [("CNN", cnn_probs), ("CNN-GRU", gru_probs)]:
            p_cond = probabilities[mask]
            pred_cond = (p_cond >= 0.5).astype(int)
            tn, fp, fn, tp = confusion_matrix(y_cond, pred_cond, labels=[0, 1]).ravel()
            condition_results.append({
                "machine": machine,
                "condition": condition,
                "model": model_name,
                "samples": int(mask.sum()),
                "accuracy": accuracy_score(y_cond, pred_cond),
                "f1": f1_score(y_cond, pred_cond, zero_division=0),
                "false_alarm_rate": fp / max(tn + fp, 1),
                "roc_auc": roc_auc_score(y_cond, p_cond) if len(np.unique(y_cond)) == 2 else np.nan
            })

condition_df = pd.DataFrame(condition_results)
display(condition_df.round(4))
condition_df.to_csv(os.path.join(PROJECT, "condition_evaluation.csv"), index=False)

## 12. Select a threshold on validation data

Choose thresholds using validation data only, then evaluate the selected threshold on the held-out test set. Avoid selecting a threshold based on test results.

In [ ]:
def get_predictions(model, loader):
    model.eval()
    labels, probabilities = [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(DEVICE))
            probabilities.extend(torch.sigmoid(logits).cpu().numpy())
            labels.extend(yb.cpu().numpy())
    return (
        np.asarray(labels).reshape(-1).astype(int),
        np.asarray(probabilities).reshape(-1)
    )


y_val, val_probs = get_predictions(cnn_gru, val_loader)
threshold_rows = []
for t in np.arange(0.05, 0.96, 0.01):
    pred = (val_probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred, labels=[0, 1]).ravel()
    threshold_rows.append({
        "threshold": round(float(t), 2),
        "accuracy": accuracy_score(y_val, pred),
        "f1": f1_score(y_val, pred, zero_division=0),
        "recall": tp / max(tp + fn, 1),
        "false_alarm_rate": fp / max(tn + fp, 1)
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df.sort_values("f1", ascending=False).head(10))

best_f1_row = threshold_df.loc[threshold_df["f1"].idxmax()]
print("Best validation F1 threshold:", best_f1_row["threshold"])
print(best_f1_row.to_dict())

max_far = 0.15
eligible = threshold_df[threshold_df["false_alarm_rate"] <= max_far]
if len(eligible):
    best_far_row = eligible.sort_values(
        ["recall", "false_alarm_rate"], ascending=[False, True]
    ).iloc[0]
    print("\nBest threshold with validation FAR <= 15%:")
    print(best_far_row.to_dict())
else:
    best_far_row = None
    print("No threshold achieved validation FAR <= 15%.")

selected_threshold = float(best_far_row["threshold"]) if best_far_row is not None else float(best_f1_row["threshold"])
print("\nSelected threshold for final test evaluation:", selected_threshold)

selected_metrics, selected_probs, selected_preds = evaluate_model(
    cnn_gru, X_test, y_test, threshold=selected_threshold
)
print("CNN-GRU test metrics at selected validation threshold:")
print(selected_metrics)
pd.DataFrame([selected_metrics]).to_csv(
    os.path.join(PROJECT, "cnn_gru_selected_threshold_metrics.csv"), index=False
)

## 13. Continuous audio inference helper

Use this on a local WAV file after training. The function breaks the recording into overlapping windows and applies smoothing/persistence. Smoothing is intended for sequential windows from the same recording, not shuffled test samples.

In [ ]:
def smooth_probabilities(probabilities, window_size=3):
    probabilities = np.asarray(probabilities, dtype=float)
    if len(probabilities) == 0:
        return probabilities
    smoothed = np.empty_like(probabilities)
    for i in range(len(probabilities)):
        start = max(0, i - window_size + 1)
        smoothed[i] = np.mean(probabilities[start:i + 1])
    return smoothed


def classify_alerts(probabilities, threshold=0.5, persistence=2):
    probabilities = np.asarray(probabilities)
    above = probabilities >= threshold
    alerts = np.zeros(len(above), dtype=int)
    count = 0
    for i, flag in enumerate(above):
        count = count + 1 if flag else 0
        alerts[i] = int(count >= persistence)
    return alerts


def analyze_continuous_audio(
    file_path, model, window_seconds=5, hop_seconds=2.5,
    threshold=0.5, smoothing_window=3, persistence=2
):
    audio, _ = librosa.load(file_path, sr=SR, mono=True)
    window_length = int(window_seconds * SR)
    hop_length = int(hop_seconds * SR)

    if len(audio) < window_length:
        audio = np.pad(audio, (0, window_length - len(audio)))

    windows = []
    for start in range(0, len(audio) - window_length + 1, hop_length):
        windows.append(waveform_to_logmel(audio[start:start + window_length]))

    features = np.stack(windows)
    probabilities = predict_probabilities(model, features)
    smoothed = smooth_probabilities(probabilities, window_size=smoothing_window)
    alerts = classify_alerts(smoothed, threshold=threshold, persistence=persistence)

    return pd.DataFrame({
        "window_start_seconds": np.arange(len(windows)) * hop_seconds,
        "abnormal_probability": probabilities,
        "smoothed_probability": smoothed,
        "alert": alerts
    })

# Example:
# recording_path = r"D:\MIMII_Dataset\sample_recording.wav"
# results = analyze_continuous_audio(
#     recording_path, cnn_gru, threshold=selected_threshold
# )
# display(results)

## 14. Outputs saved locally

The notebook saves feature arrays, metadata, split indices, model checkpoints, training histories, and evaluation CSV/NPY files in `PROJECT`. No Google Drive mount is used.

**Before long runs:** verify that the inventory cell finds WAV files. Start with the default sample cap; increase it only after checking available disk space, RAM, and processing time.